# Chuyên đề 5 — Keystone và quản lý định danh

## Notebook tổng hợp Phần 1 và Phần 2 theo lab của Tai

Notebook này tổng hợp lại các bước đã thực hành trong lab Keystone, đã thay toàn bộ thông tin mẫu `hvXX/project-hvXX` bằng thông tin thực tế của dự án:

- Controller VM: `ubuntu@192.168.150.21`
- Keystone internal endpoint: `http://10.0.0.10:5000/v3`
- Keystone public endpoint: `http://192.168.150.54:5000`
- Project: `tai_project`
- User chính: `tai_admin_cho_taiproject`
- Password ban đầu: `123.abc`
- Role chính: `member`
- Image hiện có: `Ubuntu 24.04 LTS`
- Flavor dùng được: `small`
- Network dùng được: `tai_network`

> Ghi chú: các cell code dùng cú pháp Bash. Khi copy chạy trực tiếp trên terminal thì bỏ dấu `$` nếu có trong tài liệu cũ. Trong notebook này đã bỏ dấu `$` để chạy được trực tiếp.


## PHẦN 1 — Khám phá Keystone và Identity Service

Mục tiêu phần này:

- Tạo và load file xác thực `openrc` cho user chính.
- Lấy token từ Keystone.
- Gọi API trực tiếp bằng `curl`.
- Khám phá service catalog và endpoint.
- Hiểu Domain, Project, User, Role và cơ chế phân quyền multi-tenant.


### Lab 5.1 — Đăng nhập và xác thực lần đầu

#### Bước 1 — Chuẩn bị môi trường / Load credentials

Tạo file `~/tai-openrc.sh` để OpenStack CLI biết cần xác thực tới Keystone bằng user nào, project nào và endpoint nào.


In [ ]:
# SSH vào Controller VM nếu chưa vào
ssh ubuntu@192.168.150.21

# Tạo openrc file cho user chính
cat > ~/tai-openrc.sh << EOF
export OS_AUTH_URL=http://10.0.0.10:5000/v3
export OS_PROJECT_NAME=tai_project
export OS_USERNAME=tai_admin_cho_taiproject
export OS_PASSWORD=123.abc
export OS_USER_DOMAIN_NAME=Default
export OS_PROJECT_DOMAIN_NAME=Default
export OS_IDENTITY_API_VERSION=3
EOF

# Load credentials
source ~/tai-openrc.sh

# Kiểm tra biến môi trường đã được load
env | grep OS_

# Kỳ vọng phải thấy các biến như:
# OS_AUTH_URL
# OS_PROJECT_NAME=tai_project
# OS_USERNAME=tai_admin_cho_taiproject
# OS_IDENTITY_API_VERSION=3


#### Bước 2 — Lấy token và đọc hiểu cấu trúc

Token là chuỗi xác thực tạm thời do Keystone cấp. Các dịch vụ như Nova, Neutron, Cinder dùng token này để xác định user là ai, thuộc project nào và có role gì.


In [ ]:
# Load credentials user
source ~/tai-openrc.sh

# Lấy token
openstack token issue

# Lưu token vào biến để dùng khi gọi API trực tiếp
TOKEN=$(openstack token issue -f value -c id)
echo "$TOKEN"

# Ghi chú:
# expires    : thời điểm token hết hạn
# id         : giá trị token
# project_id : ID của tai_project
# user_id    : ID của tai_admin_cho_taiproject


#### Bước 3 — So sánh token user với admin

User `tai_admin_cho_taiproject` chỉ là member trong `tai_project`, còn admin là tài khoản quản trị hệ thống. Hai token sẽ có `project_id`, `user_id` và quyền khác nhau.


In [ ]:
# Token của user chính
source ~/tai-openrc.sh
openstack token issue

# Token của admin
# Nếu bị Permission denied khi source, cần kiểm tra quyền đọc file /etc/kolla/admin-openrc.sh
source /etc/kolla/admin-openrc.sh
openstack token issue

# Quay lại user chính
source ~/tai-openrc.sh


#### Bước 4 — Gọi API trực tiếp bằng curl

Mục tiêu là chứng minh OpenStack CLI thực chất cũng lấy token rồi dùng token gọi REST API.


In [ ]:
# Load credentials user
source ~/tai-openrc.sh

# Lấy token vào biến
TOKEN=$(openstack token issue -f value -c id)

# Gọi Keystone API trực tiếp: xem project mà token có quyền truy cập
curl -s -H "X-Auth-Token: $TOKEN" http://10.0.0.10:5000/v3/auth/projects | python3 -m json.tool | head -20

# Gọi Nova API trực tiếp: xem server trong project
curl -s -H "X-Auth-Token: $TOKEN" http://10.0.0.10:8774/v2.1/servers | python3 -m json.tool

# Nếu thấy "servers": [] thì nghĩa là project chưa có VM hoặc chưa có VM active trong scope hiện tại.


### Lab 5.2 — Khám phá Service Catalog và Endpoints

Service Catalog là danh sách dịch vụ OpenStack và endpoint tương ứng. Trong lab này, user member có thể bị từ chối khi xem service list, nên dùng admin credentials để xem đầy đủ.


#### Bước 1 — Xem danh sách services


In [ ]:
# Load credentials admin
source /etc/kolla/admin-openrc.sh

# Xem tất cả service đã đăng ký
openstack service list

# Xem chi tiết service nova
openstack service show nova

# Đếm số dòng service list
openstack service list | wc -l

# Ghi chú:
# Nếu dùng user member chạy openstack service list có thể bị HTTP 403.
# Đây là hành vi đúng của Keystone RBAC.


#### Bước 2 — Phân tích endpoint

Trong dự án này public endpoint đã đổi sang `192.168.150.54`, còn internal endpoint dùng VIP nội bộ `10.0.0.10`.


In [ ]:
# Load credentials admin
source /etc/kolla/admin-openrc.sh

# Xem tất cả endpoint
openstack endpoint list

# Lọc theo service Keystone
openstack endpoint list --service keystone

# Xem endpoint public
openstack endpoint list --interface public

# Xem endpoint internal
openstack endpoint list --interface internal

# So sánh:
# public   -> 192.168.150.54:xxx  dùng từ bên ngoài/lab network
# internal -> 10.0.0.10:xxx       dùng trong mạng nội bộ control plane


#### Bước 3 — Service Catalog từ token


In [ ]:
# Load credentials user
source ~/tai-openrc.sh

# Lấy full service catalog trong token
openstack catalog list

# Xem endpoint của service Nova
openstack catalog show nova


#### Bổ sung — Dùng clouds.yaml

`clouds.yaml` là file cấu hình client OpenStack. Nó không thay đổi hệ thống, chỉ giúp CLI biết cách đăng nhập. File chuẩn nằm tại `~/.config/openstack/clouds.yaml`.


In [ ]:
# Tạo file clouds.yaml trực tiếp trên controller
cat > ~/clouds.yaml << EOF
clouds:
  openstack:
    auth:
      auth_url: http://192.168.150.54:5000
      username: "tai_admin_cho_taiproject"
      password: "123.abc"
      project_name: "tai_project"
      user_domain_name: "Default"
      project_domain_name: "Default"
    region_name: "RegionOne"
    interface: "public"
    identity_api_version: 3
EOF

# Đặt vào thư mục chuẩn
mkdir -p ~/.config/openstack
mv ~/clouds.yaml ~/.config/openstack/

# Dùng cloud name là openstack
export OS_CLOUD=openstack
openstack token issue

# Ghi chú:
# OS_CLOUD=openstack vì tên cloud trong YAML là "openstack".
# Không dùng OS_CLOUD=tai_project vì tai_project là project name, không phải cloud name.


### Lab 5.3 — Domain, Project, User và Role

Cấu trúc phân cấp Keystone:

- Domain: tổ chức cấp cao nhất, mặc định là `Default`.
- Project: không gian tách biệt tài nguyên, tương đương tenant.
- User: tài khoản người dùng.
- Role: tập quyền như `admin`, `member`, `reader`.
- Role Assignment: gán User + Role vào Project.


#### Bước 1 — Khám phá Project


In [ ]:
# Load credentials user
source ~/tai-openrc.sh

# Xem thông tin project của mình
openstack project show tai_project

# Xem project user có quyền truy cập
openstack project list
# Kỳ vọng: thấy tai_project

# Xem chi tiết domain default
openstack domain show default
# Nếu user member không có quyền xem domain thì dùng admin credentials.


#### Bước 2 — Khám phá User và Role

Một số lệnh role assignment cần quyền admin. Khi user member chạy có thể nhận HTTP 403, đây là kết quả đúng.


In [ ]:
# Load credentials user
source ~/tai-openrc.sh

# Xem thông tin user của mình
openstack user show tai_admin_cho_taiproject

# Các lệnh dưới đây có thể bị HTTP 403 với user member
openstack role assignment list --user tai_admin_cho_taiproject

openstack role assignment list   --user tai_admin_cho_taiproject   --project tai_project

openstack role list

# Nếu cần xem thật, dùng admin:
source /etc/kolla/admin-openrc.sh
openstack role assignment list --user tai_admin_cho_taiproject
openstack role assignment list --user tai_admin_cho_taiproject --project tai_project
openstack role list


#### Bước 3 — So sánh với tài khoản admin / Isolation demo


In [ ]:
# User member chỉ thấy phạm vi project của mình
source ~/tai-openrc.sh
openstack project list

# Admin thấy tất cả project
source /etc/kolla/admin-openrc.sh
openstack project list --all

# User member thử lệnh cần admin quyền
source ~/tai-openrc.sh
openstack user list
# Kỳ vọng: HTTP 403 / You are not authorized

# Kết luận:
# Multi-tenant isolation và RBAC Keystone đang hoạt động đúng.


## PHẦN 2 — Quản lý User, Project, Role và Application Credentials

Mục tiêu phần này:

- Tạo user phụ trong cùng project.
- Kiểm tra visibility tài nguyên giữa các user cùng project.
- So sánh quyền `member` và `reader`.
- Tạo và sử dụng Application Credential cho automation.
- Quản lý password và debug authentication.


### Lab 5.4 — Tạo user phụ trong project

Trong lab này dùng admin để tạo user và gán role vào project `tai_project`.


#### Bước 1 — Tạo user phụ `tai_dev`


In [ ]:
# Load credentials admin
source /etc/kolla/admin-openrc.sh

# Tạo user phụ
openstack user create   --password DevUser@2025!   --email tai_dev@lab.local   --enable   tai_dev

# Gán role member cho user phụ trong project tai_project
openstack role add   --project tai_project   --user tai_dev   member

# Xác nhận role assignment
openstack role assignment list   --project tai_project
# Kỳ vọng: thấy tai_admin_cho_taiproject và tai_dev.


#### Bước 2 — Test đăng nhập bằng user mới


In [ ]:
# Tạo openrc cho user phụ
cat > ~/tai-dev-openrc.sh << EOF
export OS_AUTH_URL=http://10.0.0.10:5000/v3
export OS_PROJECT_NAME=tai_project
export OS_USERNAME=tai_dev
export OS_PASSWORD=DevUser@2025!
export OS_USER_DOMAIN_NAME=Default
export OS_PROJECT_DOMAIN_NAME=Default
export OS_IDENTITY_API_VERSION=3
EOF

# Load và test
source ~/tai-dev-openrc.sh
openstack token issue

# Xem VM trong project
openstack server list

# Quay lại user chính
source ~/tai-openrc.sh


#### Bước 3 — Kiểm tra Data Visibility

Hai user cùng thuộc `tai_project` nên cùng nhìn thấy tài nguyên trong project đó.


In [ ]:
# Kiểm tra image, flavor, network hiện có
source ~/tai-openrc.sh
openstack image list
openstack flavor list
openstack network list

# Tạo VM bằng user chính
# Dùng đúng thông tin thực tế trong lab:
# image   : Ubuntu 24.04 LTS
# flavor  : small
# network : tai_network
openstack server create   --flavor small   --image "Ubuntu 24.04 LTS"   --network tai_network   test-visibility

# Kiểm tra server
openstack server list

# Switch sang user phụ
source ~/tai-dev-openrc.sh
openstack server list
# Kỳ vọng: thấy VM test-visibility vì cùng project tai_project

# Ghi chú:
# Nếu VM ở trạng thái ERROR, data visibility vẫn chứng minh được nếu user phụ nhìn thấy VM đó.
# ERROR thường do vấn đề scheduler/network/resource/image boot, không phải do Keystone.


### Lab 5.5 — Phân quyền role: member vs reader

Ba role mặc định thường gặp:

- `admin`: toàn quyền hệ thống.
- `member`: tạo và quản lý tài nguyên trong project.
- `reader`: chỉ đọc, không tạo/sửa/xóa tài nguyên.


#### Bước 1 — Tạo user readonly `tai_readonly` với role reader


In [ ]:
# Load credentials admin
source /etc/kolla/admin-openrc.sh

# Tạo user readonly
openstack user create   --password ReadOnly@2025!   --enable tai_readonly

# Gán role reader cho user readonly trong project
openstack role add   --project tai_project   --user tai_readonly   reader

# Xác nhận role
openstack role assignment list   --project tai_project   --user tai_readonly
# Kỳ vọng: thấy role reader.


#### Bước 2 — Test quyền reader: chỉ đọc được


In [ ]:
# Tạo openrc cho readonly user
cat > ~/tai-ro-openrc.sh << EOF
export OS_AUTH_URL=http://10.0.0.10:5000/v3
export OS_PROJECT_NAME=tai_project
export OS_USERNAME=tai_readonly
export OS_PASSWORD=ReadOnly@2025!
export OS_USER_DOMAIN_NAME=Default
export OS_PROJECT_DOMAIN_NAME=Default
export OS_IDENTITY_API_VERSION=3
EOF

# Load readonly user
source ~/tai-ro-openrc.sh

# Test đọc: phải thành công
openstack server list
openstack image list
openstack network list

# Test tạo: phải bị từ chối nếu policy áp dụng đúng role reader
openstack server create   --flavor small   --image "Ubuntu 24.04 LTS"   --network tai_network   test-readonly
# Kỳ vọng: Policy does not allow this action / HTTP 403.


#### Bước 3 — Quay lại user chính và dọn dẹp user phụ


In [ ]:
# Quay lại user chính
source ~/tai-openrc.sh

# Xóa user không cần nữa bằng admin, nếu muốn dọn lab
source /etc/kolla/admin-openrc.sh
openstack user delete tai_readonly
openstack user delete tai_dev

# Quay lại user chính
source ~/tai-openrc.sh


### Lab 5.6 — Application Credentials

Application Credential cho phép script, CI/CD, Terraform hoặc Ansible xác thực với OpenStack mà không cần dùng trực tiếp username/password. Secret chỉ hiển thị một lần khi tạo.


#### Bước 1 — Tạo Application Credential


In [ ]:
# Load credentials user chính
source ~/tai-openrc.sh

# Tạo app credential cho CI/CD
openstack application credential create   --description "CI/CD deployment credential"   tai-cicd

# LƯU LẠI SECRET NGAY vì secret chỉ hiện một lần.
# Ví dụ credential đã tạo trong lab:
APP_ID=432d5b3f6a49461d8e2e19c8d82d7e5b
APP_SECRET='W4NyGa_Ncv_GVp-DBhAQoNMuBmuFltAFR-55MFXcw3jB-1WSCVMbEdLO9QPxfB6ndBkzU-kM7hsPdTcYd6gX1Q'

# Nếu cần lấy lại ID theo name:
openstack application credential show tai-cicd -f value -c id


#### Bước 2 — Dùng Application Credential thay username/password

Khi chuyển từ password auth sang application credential auth, cần `unset` các biến scope cũ để tránh lỗi `Application credentials cannot request a scope`.


In [ ]:
# Tạo openrc dùng app credential
cat > ~/tai-appcred-openrc.sh << EOF
unset OS_PROJECT_NAME
unset OS_USERNAME
unset OS_PASSWORD
unset OS_USER_DOMAIN_NAME
unset OS_PROJECT_DOMAIN_NAME
unset OS_PROJECT_ID
unset OS_USER_ID
unset OS_TENANT_NAME
unset OS_CLOUD

export OS_AUTH_URL=http://10.0.0.10:5000/v3
export OS_AUTH_TYPE=v3applicationcredential
export OS_APPLICATION_CREDENTIAL_ID=432d5b3f6a49461d8e2e19c8d82d7e5b
export OS_APPLICATION_CREDENTIAL_SECRET='W4NyGa_Ncv_GVp-DBhAQoNMuBmuFltAFR-55MFXcw3jB-1WSCVMbEdLO9QPxfB6ndBkzU-kM7hsPdTcYd6gX1Q'
EOF

# Load app credential
source ~/tai-appcred-openrc.sh

# Test không cần username/password
openstack token issue
openstack server list


#### Bước 3 — Tạo credential có expiry và cleanup


In [ ]:
# Thoát app credential mode nếu đang dùng
unset OS_AUTH_TYPE
unset OS_APPLICATION_CREDENTIAL_ID
unset OS_APPLICATION_CREDENTIAL_SECRET

# Quay lại user chính bằng password auth
source ~/tai-openrc.sh

# Tạo app credential có thời hạn
openstack application credential create   --expiration "2026-12-31T23:59:59"   --description "Temporary credential"   tai-temp

# Xem danh sách app credentials
openstack application credential list

# Xóa credential tạm
openstack application credential delete tai-temp

# Nếu test lại bằng ID của tai-temp đã xóa thì Keystone trả lỗi:
# Could not find Application Credential ... (HTTP 404)
# Đây là kết quả đúng vì credential đã bị xóa.


### Lab 5.7 — Quản lý Password và Token Security


#### Bước 1 — Đổi password user

OpenStack CLI trong môi trường này dùng option `--original-password`, không dùng `--current-password`. Khi dùng mật khẩu có ký tự `!`, nên dùng nháy đơn trong `sed` để tránh lỗi history expansion.


In [ ]:
# Đảm bảo thoát app credential mode
unset OS_AUTH_TYPE
unset OS_APPLICATION_CREDENTIAL_ID
unset OS_APPLICATION_CREDENTIAL_SECRET

# Load user chính bằng password auth
source ~/tai-openrc.sh

# Đổi password từ 123.abc sang NewPassword@2025!
openstack user password set   --original-password "123.abc"   --password "NewPassword@2025!"

# Cập nhật openrc bằng nháy đơn để tránh lỗi dấu !
sed -i 's/123.abc/NewPassword@2025!/' ~/tai-openrc.sh

# Load lại và test
source ~/tai-openrc.sh
openstack token issue

# Đổi lại password cũ để tiện lab
openstack user password set   --original-password "NewPassword@2025!"   --password "123.abc"

# Cập nhật lại openrc về password cũ
sed -i 's/NewPassword@2025!/123.abc/' ~/tai-openrc.sh
source ~/tai-openrc.sh
openstack token issue


#### Bước 2 — Debug authentication


In [ ]:
# Debug authentication, lọc các dòng quan trọng
source ~/tai-openrc.sh
openstack --debug token issue 2>&1 | grep -E "Error|Request|Response"

# Mô phỏng lỗi sai password
OS_PASSWORD=wrongpass openstack token issue
# Kỳ vọng: 401 Unauthorized

# Mô phỏng lỗi sai endpoint
OS_AUTH_URL=http://10.0.0.10:9999 openstack token issue
# Kỳ vọng: Connection refused hoặc không kết nối được

# Xem token info chi tiết dạng JSON
openstack token issue -f json

# Xem token info chi tiết dạng YAML
openstack token issue -f yaml


## Ghi chú lỗi đã gặp và cách hiểu nhanh

- `servers: []`: API đúng, project chưa có VM.
- `No Image found`: sai tên image, phải dùng `Ubuntu 24.04 LTS`.
- `No flavor ...`: sai flavor, dùng `small` theo flavor list.
- `No Network found for private`: sai network, dùng `tai_network`.
- `HTTP 403`: user không đủ quyền, thường đúng với role `member` hoặc `reader`.
- `Application credentials cannot request a scope`: còn sót biến `OS_PROJECT_NAME`, `OS_USERNAME`, `OS_CLOUD` khi dùng app credential.
- `Could not find Application Credential (HTTP 404)`: credential đã bị xóa, đây là kết quả đúng khi test cleanup.
- `event not found` khi `sed` với password có `!`: dùng nháy đơn hoặc escape dấu `!`.
